In [30]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

In [31]:
df = pd.read_csv("data/complaints_in_just_2025.csv")

In [32]:
df.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2025-01-09,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account status incorrect,NaN,NaN,MOUNTAIN AMERICA FEDERAL CREDIT UNION,AZ,85301,NaN,Web,2025-01-27,Closed with explanation,Yes,11454251
1,2025-04-08,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account information incorrect,"To Whom It May Concern, I am filing a formal c...",NaN,MOHELA,NE,681XX,NaN,Web,2025-04-08,Closed with explanation,No,12856469
2,2025-04-15,Student loan,Federal student loan servicing,Struggling to repay your loan,"Problem with forgiveness, cancellation, or dis...",Struggling with a XXXX has significantly impac...,NaN,"Maximus Federal Services, Inc.",AL,35007,NaN,Web,2026-04-06,Closed with explanation,Yes,12989708
3,2025-04-18,Student loan,Federal student loan servicing,Dealing with your lender or servicer,Trouble with how payments are being handled,I have taken out parent loans on behalf of my ...,NaN,MOHELA,OH,43110,Servicemember,Web,2025-04-18,Closed with explanation,No,13062648
4,2025-05-05,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account information incorrect,NaN,NaN,MOHELA,OH,44224,NaN,Web,2025-05-05,Closed with explanation,No,13330457


In [33]:
df.shape

(5443005, 16)

In [34]:
df.info

<bound method DataFrame.info of         Date received                                            Product  \
0          2025-01-09  Credit reporting or other personal consumer re...   
1          2025-04-08  Credit reporting or other personal consumer re...   
2          2025-04-15                                       Student loan   
3          2025-04-18                                       Student loan   
4          2025-05-05  Credit reporting or other personal consumer re...   
...               ...                                                ...   
5443000    2025-12-31  Credit reporting or other personal consumer re...   
5443001    2025-12-31  Credit reporting or other personal consumer re...   
5443002    2025-12-31  Credit reporting or other personal consumer re...   
5443003    2025-12-31  Credit reporting or other personal consumer re...   
5443004    2025-12-31  Credit reporting or other personal consumer re...   

                            Sub-product  \
0           

In [35]:
df.isna().sum().sort_values(ascending=False)

Tags                            5302207
Consumer complaint narrative    4221035
Company public response         2275665
Sub-issue                        116668
State                              8374
Date received                         0
Product                               0
Sub-product                           0
Issue                                 0
Company                               0
ZIP code                              0
Submitted via                         0
Date sent to company                  0
Company response to consumer          0
Timely response?                      0
Complaint ID                          0
dtype: int64

In [36]:
text_col = "Consumer complaint narrative"

df_text = df[df[text_col].notna()].copy()

df_text.shape

(1221970, 16)

In [37]:
df_text["Product"].value_counts().head(10)

Credit reporting or other personal consumer reports        912758
Debt collection                                            100600
Money transfer, virtual currency, or money service          64832
Checking or savings account                                 51447
Credit card                                                 41494
Mortgage                                                    14015
Vehicle loan or lease                                       11680
Student loan                                                10881
Payday loan, title loan, personal loan, or advance loan      7800
Prepaid card                                                 3822
Name: Product, dtype: int64

In [38]:
df_text["State"].value_counts().head(10)

TX    187238
FL    146094
CA    117404
GA     99611
NY     62592
IL     57312
NC     51345
PA     38371
NJ     35388
SC     32010
Name: State, dtype: int64

In [39]:
text_col = "Consumer complaint narrative"

df_text = df[df[text_col].notna()].copy()


df_cluster = df_text[
    df_text["Product"] == "Credit reporting or other personal consumer reports"
].copy()

df_sample = df_cluster.sample(n=10000, random_state=42).copy()

df_sample.shape

(10000, 16)

In [40]:
def better_clean_text(text):
    text = str(text).lower()
    
    # remove tokens like xxxx, xx, xxxxx
    text = re.sub(r"\bx+\b", " ", text)
    
    # remove non-letters
    text = re.sub(r"[^a-z\s]", " ", text)
    
    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

df_sample["clean_text"] = df_sample["Consumer complaint narrative"].apply(better_clean_text)

In [41]:
extra_stop_words = [
    "consumer", "complaint", "credit", "report", "reports", "reporting",
    "account", "accounts", "information", "company",
    "fcra", "section", "act", "law", "states",
    "please", "attached", "xx", "xxxx"
]

custom_stop_words = list(ENGLISH_STOP_WORDS.union(extra_stop_words))

vectorizer = TfidfVectorizer(
    max_features=3000,
    stop_words=custom_stop_words,
    min_df=5,
    max_df=0.7,
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(df_sample["clean_text"])

In [42]:
k = 6

kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

df_sample["cluster"] = kmeans.fit_predict(X)

df_sample["cluster"].value_counts().sort_index()

0     342
1    1313
2     213
3     338
4    7457
5     337
Name: cluster, dtype: int64

In [43]:
cluster_counts = df_sample["cluster"].value_counts().sort_index()
cluster_percent = (cluster_counts / len(df_sample) * 100).round(2)

pd.DataFrame({
    "count": cluster_counts,
    "percent": cluster_percent
})

,count,percent
0,342,3.42
1,1313,13.13
2,213,2.13
3,338,3.38
4,7457,74.57
5,337,3.37


In [44]:
try:
    terms = vectorizer.get_feature_names_out()
except AttributeError:
    terms = vectorizer.get_feature_names()

def show_top_words(model, terms, n_words=20):
    for i, center in enumerate(model.cluster_centers_):
        top_indices = center.argsort()[::-1][:n_words]
        top_terms = [terms[j] for j in top_indices]
        print(f"\nCluster {i}:")
        print(", ".join(top_terms))


show_top_words(kmeans, terms, n_words=20)


Cluster 0:
banking, code, banking violation, efficiency banking, efficiency, violation, written consent, fair, impairs efficiency, violation impairs, impairs, says banking, banking dependent, unfair methods, directly impair, confidence essential, methods undermine, banking unfair, impair efficiency, impair

Cluster 1:
theft, identity, identity theft, inquiries, fraudulent, opened, result, date, result identity, block, hard inquiries, fraud, collections, date opened, hard, number, theft fraud, items, opened balance, appearing

Cluster 2:
code, entries damaging, damaging unjust, unjust especially, correct pursue, especially ive, requiring proper, ive opened, fail investigate, unauthorized violation, accuracy false, mandates maximum, false entries, action fdcpa, authorized fail, code deceptive, items mandates, fdcpa code, deceptive misleading, code requiring

Cluster 3:
agency, person, furnish, privacy, usc, violated, person knows, agency person, knows, shall, knows reasonable, cause bel

In [45]:
cluster_summary = (
    df_sample["cluster"]
    .value_counts()
    .sort_index()
    .reset_index()
)

cluster_summary.columns = ["cluster", "count"]
cluster_summary["percent"] = (cluster_summary["count"] / len(df_sample) * 100).round(2)

cluster_summary

,cluster,count,percent
0,0,342,3.42
1,1,1313,13.13
2,2,213,2.13
3,3,338,3.38
4,4,7457,74.57
5,5,337,3.37


In [47]:
cluster_examples = (
    df_sample
    .groupby("cluster", group_keys=False)
    .apply(lambda x: x.sample(n=min(5, len(x)), random_state=42))
    [["cluster", "Product", "Issue", "Sub-issue", "Consumer complaint narrative"]]
)

cluster_examples.head()

,cluster,Product,Issue,Sub-issue,Consumer complaint narrative
3552121,0,Credit reporting or other personal consumer re...,Improper use of your report,Reporting company used your report improperly,I am writing to formally file a complaint rega...
4523601,0,Credit reporting or other personal consumer re...,Incorrect information on your report,Information is missing that should be on the r...,The Fair Credit Reporting Act ( 15 U. S. Code ...
4475058,0,Credit reporting or other personal consumer re...,Improper use of your report,Reporting company used your report improperly,The Fair Credit Reporting Act ( 15 U.S. Code 1...
1074255,0,Credit reporting or other personal consumer re...,Incorrect information on your report,Information belongs to someone else,The Fair Credit Reporting Act ( 15 U.S. Code 1...
4041010,0,Credit reporting or other personal consumer re...,Improper use of your report,Reporting company used your report improperly,The Fair Credit Reporting Act ( 15 U.S. Code 1...


This is an initial clustering test using 10,000 sampled 2025 credit reporting complaint narratives. I used temporary basic text cleaning, TF-IDF, and KMeans with 6 clusters.

The first result shows one large general cluster and several smaller clusters. Some smaller clusters appear to be driven by repeated legal/template language rather than clear complaint themes.  Once the NLP preprocessing output is available, I will rerun the same clustering workflow and compare whether the clusters become more balanced and interpretable.